<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/04a_ragas_aspect_critic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 4a: Custom LLM-as-a-Judge: RAGAS AspectCritic

**Goal:** Build a custom RAGAS AspectCritic evaluator using Claude as the
judge model. AspectCritic evaluates responses against named, domain-specific
aspects rather than generic RAG quality metrics. Here the aspects are drawn
directly from the EU AI Act and NIST AI RMF obligations established in the
Phase 3b G-Eval rubrics.

**Tools:** RAGAS AspectCritic, Claude (claude-sonnet-4-6) as judge

**Aspects evaluated:**
- correctness: factual accuracy relative to retrieved regulatory documents
- regulatory_grounding: claims traceable to specific regulatory articles
- oversight_representation: Article 14 human oversight accurately represented
- bias_representation: Article 10 data governance accurately represented
- harm_potential: does the response risk misleading a deployer about compliance

**Design addition (Federico Blanco Sanchez-Llanos):** The two-queue split
(quality failures route to retrieval/generation layer, compliance failures
route to governance layer) must survive independent of whoever made the call.
This notebook exports each AspectCritic verdict as a signed artifact bound
to a hash of the specific inputs so the routing decision is independently
auditable.

**SIMULATED_OUTPUT flag:** Set to True throughout.

**Date:** July 2026

In [1]:
# Cell 2: Mount Drive and confirm prior phases

from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

phase3b_path = DRIVE_PATH + "phase03b_governance_metrics_results.json"
if os.path.exists(phase3b_path):
    with open(phase3b_path) as f:
        phase3b = json.load(f)
    print("Phase 3b results confirmed.")
    print(f"  Outcome accuracy: {phase3b['overall']['outcome_accuracy']}")
    print(f"  Adversarial detection: "
          f"{phase3b['overall']['adversarial_detection_rate']}")
    print(f"  Artifact limitation: "
          f"{phase3b['governance_evaluation']['artifact_limitation'][:80]}...")
else:
    print("WARNING: Phase 3b results not found.")
    print(f"Expected: {phase3b_path}")
    print("Run 03b_deepeval_governance_metrics.ipynb first.")

Mounted at /content/drive
Phase 3b results confirmed.
  Outcome accuracy: 6/7
  Adversarial detection: 3/3
  Artifact limitation: All compliance verdicts are bound to SHA-256 input hashes. Hashes prove non-alte...


In [2]:
# Cell 3: Install packages

!pip install ragas==0.3.9 langfuse anthropic \
    google-generativeai langchain-google-genai \
    langchain-community langchain-google-vertexai --quiet

print("Packages installed.")
print("ragas==0.3.9 (pinned: avoids broken VertexAI import in 0.4.x)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.7/366.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 669.4/669.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.9/118.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.

In [3]:
# Cell 4: Simulated output flag, clients, and thresholds

SIMULATED_OUTPUT = True

JUDGE_MODEL = "claude-sonnet-4-6"

from google.colab import userdata

if not SIMULATED_OUTPUT:
    import anthropic
    claude_client = anthropic.Anthropic(
        api_key=userdata.get('ANTHROPIC_API_KEY')
    )
    from langfuse import Langfuse
    langfuse = Langfuse(
        public_key=userdata.get('LANGFUSE_PUBLIC_KEY'),
        secret_key=userdata.get('LANGFUSE_SECRET_KEY'),
        host="https://cloud.langfuse.com"
    )
    print("Claude client initialised.")
    print("Langfuse client initialised.")
else:
    print("[SIMULATED] Clients not initialised.")
    print(f"SIMULATED_OUTPUT = {SIMULATED_OUTPUT}")
    print(f"Judge model: {JUDGE_MODEL}")

# Routing thresholds consistent across all phases
PASS_THRESHOLD = 0.80
FAIL_THRESHOLD = 0.60

# AspectCritic verdict mapping
# RAGAS AspectCritic returns binary verdicts per aspect.
# We map to scores for consistent Langfuse logging.
VERDICT_SCORES = {
    "yes": 1.0,   # aspect satisfied
    "no":  0.0    # aspect not satisfied
}

print()
print("Routing thresholds:")
print(f"  >= {PASS_THRESHOLD}: PASS      -> quality layer")
print(f"  <  {FAIL_THRESHOLD}: FAIL      -> governance layer")
print(f"  between:    BORDERLINE -> human review queue")
print()
print("AspectCritic verdict scores:")
print(f"  yes (aspect satisfied):     1.0")
print(f"  no  (aspect not satisfied): 0.0")

[SIMULATED] Clients not initialised.
SIMULATED_OUTPUT = True
Judge model: claude-sonnet-4-6

Routing thresholds:
  >= 0.8: PASS      -> quality layer
  <  0.6: FAIL      -> governance layer
  between:    BORDERLINE -> human review queue

AspectCritic verdict scores:
  yes (aspect satisfied):     1.0
  no  (aspect not satisfied): 0.0
